In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
from dataclasses import dataclass
import math

In [ ]:
class Tile(Enum):
    EMPTY = 0

    BELT_UP = 1
    BELT_RIGHT = 2
    BELT_DOWN = 3
    BELT_LEFT = 4

    BELT_RIGHT_UP = 5
    BELT_DOWN_RIGHT = 6
    BELT_LEFT_DOWN = 7
    BELT_UP_LEFT = 8 

    BELT_UP_RIGHT = 9
    BELT_RIGHT_DOWN = 10
    BELT_DOWN_LEFT = 11
    BELT_LEFT_UP = 12

    INSERTER_UP = 13
    INSERTER_RIGHT = 14
    INSERTER_DOWN = 15
    INSERTER_LEFT = 16

    @staticmethod
    def INSERTERS():
        return [Tile.INSERTER_UP,
                Tile.INSERTER_RIGHT,
                Tile.INSERTER_DOWN,
                Tile.INSERTER_LEFT]
    
    @staticmethod
    def ALL():
        return [i for i in Tile]

    @staticmethod
    def BELTS():
        return [Tile(i) for i in range(1,12+1)]
    
class Direction(Enum):
    DIR_NONE = 0
    UP = 1
    RIGHT = 2
    DOWN = 3
    LEFT = 4
    

DIR_NONE = Direction.DIR_NONE
UP = Direction.UP
RIGHT = Direction.RIGHT
DOWN = Direction.DOWN
LEFT = Direction.LEFT

RelPos = tuple[int,int] # x (right+), y(up+) 

class TileData:
    def __init__(self,output:RelPos,can_give_to:list[Tile]):
        self._output = output
        self._can_give_to = can_give_to

        self._pull_pos = None
        self._pull_max_th = 1

    def pull(self,pos:RelPos,max_th=1):
        self._pull_pos = pos
        self._pull_max_th = max_th

        return self
    
    def can_pull(self):
        return self._pull_pos is not None
    
    @property
    def can_give_to(self):
        return [i.value for i in self._can_give_to]
    
TILE_DATA: dict[Tile, TileData] = {
    Tile.EMPTY:
        TileData((0, 0), Tile.ALL()),

    Tile.BELT_UP:
        TileData((0, 1), [
            Tile.BELT_UP,
            Tile.BELT_DOWN_LEFT,
            Tile.BELT_DOWN_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_RIGHT:
        TileData((1, 0), [
            Tile.BELT_RIGHT,
            Tile.BELT_LEFT_UP,
            Tile.BELT_LEFT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_DOWN:
        TileData((0, -1), [
            Tile.BELT_DOWN,
            Tile.BELT_UP_LEFT,
            Tile.BELT_UP_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_LEFT:
        TileData((-1, 0), [
            Tile.BELT_LEFT,
            Tile.BELT_RIGHT_UP,
            Tile.BELT_RIGHT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_RIGHT_UP:
        TileData((0, 1), [
            Tile.BELT_UP,
            Tile.BELT_DOWN_LEFT,
            Tile.BELT_DOWN_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_DOWN_RIGHT:
        TileData((1, 0), [
            Tile.BELT_RIGHT,
            Tile.BELT_LEFT_UP,
            Tile.BELT_LEFT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_LEFT_DOWN:
        TileData((0, -1), [
            Tile.BELT_DOWN,
            Tile.BELT_UP_LEFT,
            Tile.BELT_UP_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_UP_LEFT:
        TileData((-1, 0), [
            Tile.BELT_LEFT,
            Tile.BELT_RIGHT_UP,
            Tile.BELT_RIGHT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_UP_RIGHT:
        TileData((1, 0), [
            Tile.BELT_RIGHT,
            Tile.BELT_LEFT_UP,
            Tile.BELT_LEFT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_RIGHT_DOWN:
        TileData((0, -1), [
            Tile.BELT_DOWN,
            Tile.BELT_UP_LEFT,
            Tile.BELT_UP_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_DOWN_LEFT:
        TileData((-1, 0), [
            Tile.BELT_LEFT,
            Tile.BELT_RIGHT_UP,
            Tile.BELT_RIGHT_DOWN,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.BELT_LEFT_UP:
        TileData((0, 1), [
            Tile.BELT_UP,
            Tile.BELT_DOWN_LEFT,
            Tile.BELT_DOWN_RIGHT,
            Tile.EMPTY,
        ] + Tile.INSERTERS()),

    Tile.INSERTER_UP:
        TileData((0, 1), Tile.ALL()).pull((0, -1), 0.2),

    Tile.INSERTER_RIGHT:
        TileData((1, 0), Tile.ALL()).pull((-1, 0), 0.2),

    Tile.INSERTER_DOWN:
        TileData((0, -1), Tile.ALL()).pull((0, 1), 0.2),

    Tile.INSERTER_LEFT:
        TileData((-1, 0), Tile.ALL()).pull((1, 0), 0.2),
}

In [ ]:
BOX_SIZE = 15
INPUT_POS = (14,2)
NEG_INF = -500.0

In [ ]:
def plot_solve(solve):
    fig, ax = plt.subplots(1,solve.size(-1),figsize=(6*5, 6))
    for i, grid in enumerate(solve.permute(2,0,1)):
        ax[i].matshow(grid.detach().exp().numpy(),vmin=0,vmax=1)
        ax[i].set
        ax[i].set_title(Tile(i))
    plt.show()

def cawr(x):
    if x<0.5:
        return np.maximum( np.cos(x*2*np.pi/2)**2 + np.cos(x*2*np.pi*5 % (np.pi/2))/5 , 0.001)
    else:
        return 0.2*(1-(x-0.5)*2)
    # return ( np.cos(x*np.pi/2)**2)
    # return np.cos(x*np.pi*5 % (np.pi/2))/5

    # if x<0.8:
    #     return max(0.4, (1 - x /4))
    # else:
    #     return max(0.2, 0.4 - (x-0.8) /0.2)
 
g=list(map(cawr, np.arange(0,1501)/1501))
print(min(g),max(g))
plt.plot(g)
plt.show()

In [ ]:
def log1m_sum_exp(log_prob, dim=-1):
    m = log_prob.max(dim=dim, keepdim=True).values
    log_sum_exp = m + torch.log(torch.exp(log_prob - m).sum(dim=dim, keepdim=True))

    p_sum = torch.exp(log_sum_exp)

    eps = torch.finfo(log_prob.dtype).eps
    p_sum = torch.clamp(p_sum, max=1 - eps)

    return torch.log1p(-p_sum).squeeze(dim)

In [ ]:
def eval(solve, verbose=0,max_iter=BOX_SIZE*BOX_SIZE):
    items_old = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), NEG_INF)
    items_old[INPUT_POS][:] = 0.0

    for iter in range(round(max_iter)):
        log_total_mass = items_old.max(dim=-1)[0]
        log_total_mass = torch.clamp(log_total_mass, min=NEG_INF)

        log_pull = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), NEG_INF, device=solve.device)
        for i in range(len(Tile)):
            tile_type = Tile(i)
            t_data = TILE_DATA[tile_type]
            if not t_data.can_pull():
                continue

            inp_y = slice(max(t_data._pull_pos[1],0), BOX_SIZE-max(-t_data._pull_pos[1],0))
            inp_x = slice(max(-t_data._pull_pos[0],0), BOX_SIZE-max(t_data._pull_pos[0],0))

            out_y = slice(max(-t_data._pull_pos[1],0), BOX_SIZE-max(t_data._pull_pos[1],0))
            out_x = slice(max(t_data._pull_pos[0],0), BOX_SIZE-max(-t_data._pull_pos[0],0))

            log_pull[out_y, out_x, i] = solve[inp_y, inp_x, i] + np.log(t_data._pull_max_th)

        log_total_demand = torch.logsumexp(log_pull, dim=-1)
        
        log_norm_scale = -torch.clamp(log_total_demand - log_total_mass, min=0.0) # [Y,X] % of mass that will be pulled

        log_pulled_mass = log_pull + log_norm_scale.unsqueeze(-1) # [Y,X,T]

        log_fraction_pulled = log_pulled_mass - log_total_mass.unsqueeze(-1)
        log_self_fraction = log1m_sum_exp(log_fraction_pulled)

        mul_self = items_old + solve + log_self_fraction.unsqueeze(-1)

        items = torch.full((BOX_SIZE, BOX_SIZE, len(Tile)), NEG_INF)
        for i in range(len(Tile)):
            tile_type = Tile(i)
            t_data: TileData = TILE_DATA[tile_type]

            if len(t_data._output) != 0:
                idx = t_data.can_give_to
                if not t_data.can_pull():
                    inp_y = slice(max(t_data._output[1],0), BOX_SIZE-max(-t_data._output[1],0))
                    inp_x = slice(max(-t_data._output[0],0), BOX_SIZE-max(t_data._output[0],0))

                    out_y = slice(max(-t_data._output[1],0), BOX_SIZE-max(t_data._output[1],0))
                    out_x = slice(max(t_data._output[0],0), BOX_SIZE-max(-t_data._output[0],0))

                    src = mul_self[inp_y,inp_x, i].unsqueeze(-1).repeat(1, 1, len(idx))
                    items[out_y, out_x, idx] = torch.logaddexp(items[out_y, out_x, idx], src)
                else:
                    dy = t_data._output[1]-t_data._pull_pos[1]
                    dx = t_data._output[0]-t_data._pull_pos[0]

                    inp_y = slice(max(dy,0), BOX_SIZE-max(-dy,0))
                    inp_x = slice(max(-dx,0), BOX_SIZE-max(dx,0))

                    out_y = slice(max(-dy,0), BOX_SIZE-max(dy,0))
                    out_x = slice(max(dx,0), BOX_SIZE-max(-dx,0))

                    src = (log_pulled_mass[inp_y, inp_x, i]).unsqueeze(-1).repeat(1, 1, len(idx))
                    items[out_y,out_x, idx] = torch.logaddexp(items[out_y,out_x, idx], src)
        if verbose>5:
            plt.matshow(items.detach().max(-1)[0].exp().numpy(),vmin=0,vmax=1)
            plt.show()
        
        sum_items = items.max(-1)[0].exp().sum().item()
        assert sum_items <= 1.15, f'Sum should be <= 1. ({sum_items})'
        
        items_old = items.clone()
    return items

# eval(solve,verbose=10)

In [ ]:
logits = torch.full((BOX_SIZE,BOX_SIZE,len(Tile)), -5000.0)
logits[:,:,Tile.EMPTY.value] = 0


logits[14, 2, Tile.BELT_RIGHT.value] = 10000
logits[14, 3, Tile.BELT_RIGHT.value] = 10000
logits[13, 2, Tile.INSERTER_UP.value] = 10000
logits[13, 3, Tile.INSERTER_UP.value] = 10000
logits[12, 2, Tile.BELT_RIGHT.value] = 10000
logits[12, 3, Tile.BELT_RIGHT.value] = 10000
logits[12, 4, Tile.BELT_LEFT_UP.value] = 10000
# logits[14,2][Tile.EMPTY.value] = -5000.0
# logits[14,2][Tile.BELT_UP.value] = 5.0 # Feed from bottom
solve = torch.log_softmax(logits, dim=-1)
plot_solve(solve)
items = eval(solve,verbose=10,max_iter=4)
plt.matshow(items.detach().max(-1)[0].exp().numpy(),vmin=0,vmax=1)
plt.show()
print(f'Output pulled: {items[11,4].max(-1)[0].exp()}, Output left: {items[14,4].max(-1)[0].exp()}')

In [ ]:
logits = torch.full((BOX_SIZE,BOX_SIZE,len(Tile)), -5000.0)
active_tiles = list(range(0,16+1))

logits[:,:,active_tiles] = 5 * torch.randn(BOX_SIZE,BOX_SIZE,len(active_tiles))

logits.requires_grad_()
# solve = smax(solve)

MAX_ITER = 101
START_TEMP = 100

opt = torch.optim.Adam([logits], lr=5)


for iter in range(MAX_ITER):
    temp = cawr(min(1,iter/50))
    # # with torch.no_grad():

    logits_noise = logits + 10 * temp * torch.randn_like(logits)

    solve = torch.nn.functional.log_softmax(logits_noise/(START_TEMP*max(temp,0.01)),dim=-1)
    # plot_solve(solve)
    # with torch.no_grad():
    

    solve = torch.clamp(solve, max=-0.0001)

    if iter%10==0:
        plot_solve(solve)
    items = eval(solve,max_iter=BOX_SIZE*5)
    # path_loss = -items[0, 13, :].mean(dim=-1)
    path_loss = -items[0, 13, :].max(dim=-1)[0]
    
    # cost_loss = -( torch.logsumexp(solve[:,:,0], (0,1))/BOX_SIZE**2 )**2 * 1500
    # cost_loss = ( torch.logsumexp(solve[:,:,1::], (0,1,2))/BOX_SIZE**2/(len(Tile)-1))**2 * 1500 * 500 * 3

    inserter_penalty = (logits_noise[:,:,[i.value for i in Tile.INSERTERS()]]**2).mean() * 3000 # * 3.5
    belt_penalty = (logits_noise[:,:,[i.value for i in Tile.BELTS()]]**2).mean() * 1
    
    # cost_loss = -solve[:,:,0].mean() 
    # l2_non_empty = (logits[:, :, 1:] ** 2).mean() * 0.005
    # loss += 3* ((1-solve.sum(dim=-1))**2).mean()


    # print(f"Grad norm: {grad_norm:.4f} | Empty cells: {empty_ratio:.1%}")
    # print(loss.grad_fn)
    
    # if loss.item()<0.05:
    #     break
    loss = path_loss + (belt_penalty + inserter_penalty) * 0.01
    loss.backward()
    # print(logits.grad)
    opt.step()

    grad_norm = logits.grad.abs().mean().item()
    empty_ratio = (solve[:, :, 0] > -0.5).float().mean().item()
    print(loss.item(), path_loss.item(),belt_penalty.item(),inserter_penalty.item(),temp, grad_norm, empty_ratio)

    opt.zero_grad()


plot_solve(solve)

In [ ]:
solve = torch.log_softmax(logits/10, dim=-1)

In [ ]:
import os
from PIL import Image
from pathlib import Path
assets = Path('assets').iterdir()
assets = {k.name[0:-4]:Image.open(k) for k in assets}

In [ ]:
# for i in assets.values():
#     i.close()

In [ ]:
def render_solve(solve: torch.Tensor):
    fig, axes = plt.subplots(BOX_SIZE, BOX_SIZE, figsize=(8, 8),gridspec_kw={"wspace": 0, "hspace": 0})

    for ax, v in zip(axes.ravel(), solve.reshape(BOX_SIZE*BOX_SIZE,-1)):
        tile = Tile(v.argmax().item())
        # print(tile)
        img = assets.get(tile.name.lower())
        if img:
            ax.imshow(img)
        ax.axis("off")
    plt.show()
render_solve(solve)

In [ ]:
eval(solve,verbose=10,max_iter=15)